# Trinity Risk Guard — EXL Challenge 04

**可运行的 Colab 技术演示：** 完成 R→Python 真实迁移测试、Performance/Stability、模拟 Fairness 方法检查、最后接入真实 AI Agent。

> ⚠️ **所有数据都是模拟的，系数并非银行实证估计；没有运行 R 不可宣称 Parity PASS，没有实际 LLM 调用不可宣称 Agent 已运行。** 代码由 ChatGPT 辅助生成，应由小组审核。

运行前：从聊天中下载 `Trinity_Risk_Guard_Technical_v3.zip`，在下面第一个单元格上传。

In [ ]:
from pathlib import Path
import os, sys, zipfile, subprocess, shutil, json

# In Google Colab, upload the full technical ZIP; in local Jupyter, open
# this notebook from the extracted project and skip upload automatically.
if not Path('requirements.txt').exists():
    try:
        from google.colab import files
        uploaded = files.upload()  # select Trinity_Risk_Guard_Technical_v3.zip
        archives = [Path(x) for x in uploaded if x.endswith('.zip')]
        if not archives:
            raise ValueError('Please upload the Trinity Risk Guard .zip file.')
        with zipfile.ZipFile(archives[0]) as bundle:
            bundle.extractall('/content')
        os.chdir('/content/Trinity_Risk_Guard_Technical_v3')
    except ImportError:
        raise RuntimeError('Extract project ZIP, then start notebook from its root.')
print('Working directory:', Path.cwd())
print('README available:', Path('README.md').exists())

## 1. Python 依赖与 R 环境（运行 R 是正式 Parity 的前提）

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
if shutil.which('Rscript'):
    print('Rscript found:', shutil.which('Rscript'))
else:
    print('Rscript not found. Installing base R on Colab Linux...')
    try:
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','r-base-core'],check=True)
    except Exception as error:
        print('R installation unsuccessful. Real cross-language parity remains NOT_RUN:',error)
print('Current Rscript:',shutil.which('Rscript'))

## 2. 运行完整的 R → Python 迁移、Parity 与模型验证

In [ ]:
command = [sys.executable, 'run_pipeline.py']
if shutil.which('Rscript'):
    command.append('--require-r')
subprocess.run(command,check=True)
metrics = json.loads(Path('outputs/metrics.json').read_text(encoding='utf-8'))
print('PARITY:', metrics['cross_language_parity']['status'])
print('AUC:', round(metrics['performance']['overall']['auc'],4))
print('BRIER:',round(metrics['performance']['overall']['brier'],4))
print('PSI:',round(metrics['stability']['utilization_psi'],4))
print('FAIRNESS:',metrics['fairness_method_demonstration']['disclaimer'])

## 3. 检查代码测试及模拟数据

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'],check=True)
import pandas as pd
from IPython.display import display
display(pd.read_csv('synthetic_credit_data.csv').head(8))

## 4. 手动输入一位 **模拟借款人** 的四个变量，检查 Python PD 输出

输入仅用于演示，不要放真实个人贷款资料或任何客户信息。

In [ ]:
from scorecard_port import score_frame
sample = pd.DataFrame([{
    'customer_id':'COLAB-DEMO-001',
    'utilization':0.30,            # 30% utilization
    'dti':0.32,                    # 32% debt-to-income
    'income_k':50.0,              # EUR 50k/year
    'late_payments_12m':0
}])
display(score_frame(sample))

## 5. 真实 LLM Documentation Agent（需 API Key 或兼容本地 LLM）

该部分与上面的 Logistic Python 评分不同：LLM 会选择工具，读取 R 模型源码与计算指标，并产生**待人工审核**的模型文档；会留下工具调用 trace。请使用个人或团队有权使用的模型 API，可能产生费用。不要公开 API Key，不要上传真实客户数据。

**没有 API Key 可跳过下面两个单元格，届时 Agent 状态应保持 NOT_RUN。**

In [ ]:
import getpass
# DO NOT paste API keys in a notebook saved/shared with others.
model_choice = input('Compatible tool-calling model [gpt-4.1-mini]: ').strip() or 'gpt-4.1-mini'
os.environ['TRG_MODEL'] = model_choice
secret = getpass.getpass('API key (hidden input, leave empty to skip): ').strip()
if secret:
    os.environ['TRG_API_KEY'] = secret
    print('Credentials configured for current session only.')
else:
    print('No API key; skip the next cell. This is NOT a live Agent result.')

In [ ]:
if os.environ.get('TRG_API_KEY') or os.environ.get('OPENAI_API_KEY') or os.environ.get('TRG_BASE_URL'):
    subprocess.run([sys.executable,'documentation_agent.py','--model-id','CRD-001'], check=True)
    status=json.loads(Path('outputs/agent_status.json').read_text(encoding='utf-8'))
    print('Agent status:',status['status'])
    if status['status']=='LIVE_EXECUTED_DRAFT_PENDING_REVIEW':
        doc=json.loads(Path('outputs/agent_documentation_draft.json').read_text(encoding='utf-8'))
        print('Draft title:',doc['summary'])
        print('Manual approval:',doc['reviewer_decision'])
        trace=json.loads(Path('outputs/agent_trace.json').read_text(encoding='utf-8'))
        print('Tools called:',[e['tool'] for e in trace['tool_events']])
else:
    print('NOT_RUN: live LLM connection not configured.')

## 6. 导出证据供小组检查

In [ ]:
from pathlib import Path
import zipfile
with zipfile.ZipFile('Trinity_Risk_Guard_Evidence.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in Path('outputs').glob('*'):
        if f.is_file():z.write(f,arcname=str(f))
print('Saved Trinity_Risk_Guard_Evidence.zip. Review R parity, Agent trace and all notes before submitting.')
try:
    from google.colab import files
    files.download('Trinity_Risk_Guard_Evidence.zip')
except ImportError:
    print('Evidence ZIP is saved in this working directory.')